# AIX360 Implementation
This notebook demonstrates AI Explainability 360 (AIX360) on the Medical Insurance dataset.

## Requirements
Run the following to install required dependencies. See `requirements-xai-frameworks.txt` for details.\n**CRITICAL**: AIX360 sometimes requires older scikit-learn (1.3.2) and numpy (<2.0).

In [ ]:
!pip install aix360==0.3.0 scikit-learn==1.3.2 numpy<2.0 pandas

## Setup and Data
Loading the Medical Insurance dataset (dummy tabular data).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# Dummy data generator for AIX360.
# Note: AIX360 has strong classification algorithms (Boolean Rule CG, Contrastive Explanations). 
# We'll frame insurance charges as High/Low (Classification) for demonstration.
np.random.seed(42)
df = pd.DataFrame({
    'age': np.random.randint(18, 65, 1000),
    'sex': np.random.choice([0, 1], 1000),           # 0=female, 1=male
    'bmi': np.random.uniform(18, 40, 1000),
    'children': np.random.randint(0, 5, 1000),
    'smoker': np.random.choice([0, 1], 1000),        # 0=no, 1=yes
    'region': np.random.choice([0, 1, 2, 3], 1000),
    'charges': np.random.uniform(1000, 50000, 1000)
})

# Classify as High (1) if charges > 25000, else Low (0)
df['high_charge'] = (df['charges'] > 25000).astype(int)

features = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
X = df[features]
y = df['high_charge']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print("Data loaded and Classification model trained!")


## Explanability Framework API
Implementing XAI Platform internal API conventions for AIX360: `create_explainer`, `compute_global_aix360`, and `explain_instance`.

In [ ]:
# Note: Due to complex dependency environments (TF/Keras/Solvers), 
# this demonstrates the structure using simpler wrappers or stubs.
# Real AIX360 requires a running environment.
class AIX360ServiceStub:
    @staticmethod
    def create_explainer(model, X_train):
        # e.g., using CEM (Contrastive Explanation Method) explainer setup
        return {"model": model, "X_bg": X_train}
        
    @staticmethod
    def compute_global_aix360(explainer):
        # Global rules, e.g., BooleanRuleCG
        print("Computing AIX360 Global Rules...")
        return {"rules": [{"rule": "smoker > 0.5 AND age > 40", "prediction": "High Charge"}]}
        
    @staticmethod
    def explain_instance(explainer, instance):
        # Local Contrastive Explanations: 
        # "If BMI was 22 instead of 30, the charge would have been Low"
        print("Computing Local Contrastive Explanation (CEM)...")
        return {"pertinent_positive": instance.values, "pertinent_negative": instance.values * 0.9}

service = AIX360ServiceStub()
explainer = service.create_explainer(model, X_train)


## Global Explanation

In [ ]:
global_exp = service.compute_global_aix360(explainer)
print("AIX360 Global Explanation Rules:")
print(global_exp)


## Local Explanation

In [ ]:
sample_instance = X_test.iloc[0:1]
local_exp = service.explain_instance(explainer, sample_instance)
print("AIX360 Local Contrastive Explanation:")
print(local_exp)
